# 01 – Full load from PostgreSQL into the lake (`replace`)

**Goal (Task 2, part 2):** copy **all 4 tables** of the PawHome database (`shelters`, `adopters`, `animals`, `applications`) into the Silo lake as **Parquet** files.

**Mode `replace`** = every run throws away the previous copy of a table in the lake and writes a fresh, complete one.

Every row gets these **technical metadata** columns:

| column | meaning |
|---|---|
| `_source_system` | where the data comes from → `postgres` |
| `_source_resource` | which table → e.g. `animals` |
| `_ingestion_type` | `full` |
| `_ingested_at` | when this run happened (UTC) |
| `_load_id` | the dlt id of this run (also in the file name) |

Object keys look like: `postgres/animals/ingestion_date=2026-10-24/<load_id>.<file_id>.parquet`

In [ ]:
# Setup: load the shared helpers from lake.py (same folder)
import dlt
from dlt.sources.sql_database import sql_database
from lake import (POSTGRES_URL, DB_TABLES, PIPELINES_DIR,
                  lake_destination, add_metadata, list_lake, read_lake)

print("dlt version:", dlt.__version__)
print("Database:", POSTGRES_URL)

## Step 0 – Check part 1 (the files you uploaded in the Silo UI)

You created the bucket `pawhome-lake` and uploaded a file twice (once at the root, once with a key prefix) in the Silo console. This lists **everything** in the bucket:

In [ ]:
list_lake("", only_parquet=False)

## Step 1 – Describe the source

`sql_database(...)` connects to PostgreSQL, reads the table definitions and creates one dlt **resource** per table.
Then `add_metadata(...)` (from `lake.py`) adds the 5 metadata columns to every row of every table.

In [ ]:
def build_full_source():
    source = sql_database(POSTGRES_URL, table_names=DB_TABLES)
    for name in DB_TABLES:
        add_metadata(source.resources[name],
                     source_system="postgres",
                     source_resource=name,
                     ingestion_type="full")
    return source

## Step 2 – Create the pipeline and run it

- `destination=lake_destination("silo")` → write into the Silo bucket, with our key layout
- `dataset_name="postgres"` → the first folder of every key (the source system)
- `write_disposition="replace"` → full load
- `loader_file_format="parquet"` → Parquet files

In [ ]:
full_pipeline = dlt.pipeline(
    pipeline_name="pawhome_full",
    destination=lake_destination("silo"),
    dataset_name="postgres",
    pipelines_dir=PIPELINES_DIR,
)

info = full_pipeline.run(build_full_source(), write_disposition="replace", loader_file_format="parquet")
print(info)
print("Rows per table:", full_pipeline.last_trace.last_normalize_info.row_counts)

## Step 3 – Look at the lake

One Parquet file per table, with the required key format.

In [ ]:
full_keys = list_lake("postgres")
full_keys = full_keys[full_keys.key.str.match(r"postgres/(shelters|adopters|animals|applications)/")]
full_keys

In [ ]:
animals = read_lake("postgres/animals/")
print(animals.shape[0], "rows")
animals[["id", "name", "species", "status", "_source_system", "_source_resource",
         "_ingestion_type", "_ingested_at", "_load_id"]].head()

## Step 4 – Run it again and compare

Because the mode is `replace`, the **old files disappear** and **new ones** (with a new `load_id`) take their place. The number of rows stays the same (as long as nobody changed the database in between).

In [ ]:
before = set(full_keys.key)

info = full_pipeline.run(build_full_source(), write_disposition="replace", loader_file_format="parquet")
print("Rows per table:", full_pipeline.last_trace.last_normalize_info.row_counts)

after_df = list_lake("postgres")
after = set(after_df[after_df.key.str.match(r"postgres/(shelters|adopters|animals|applications)/")].key)

print("\nREMOVED (old run):")
for k in sorted(before - after): print("  -", k)
print("\nADDED (new run):")
for k in sorted(after - before): print("  +", k)

In [ ]:
animals = read_lake("postgres/animals/")
print(animals.shape[0], "rows, load ids in the lake:", animals["_load_id"].unique())

**What we see:** after the rerun each table still has exactly **one** file, its `load_id` is new, and the row count is unchanged. A full `replace` load always gives a fresh complete copy and never duplicates data, but it re-reads **everything** every time (slow for big tables). That is why part 3 uses incremental loading.